# Taller 2 - Punto 2: Embeddings de Oraciones con Sentence Transformers

Este notebook implementa el procesamiento de documentos PDF y la generación de embeddings de oraciones usando diferentes modelos de Sentence Transformers.

## Objetivos:
1. Carga y procesamiento de documentos PDF
2. Fragmentación (chunking) del texto
3. Generación de embeddings con 4 modelos diferentes
4. Consulta de similitud semántica usando similitud coseno
5. Visualización 2D con PCA para cada modelo

## 2.1 Instalación de Dependencias

In [ ]:
!pip install sentence-transformers
!pip install langchain
!pip install langchain-community
!pip install pymupdf
!pip install pypdf
!pip install scikit-learn
!pip install matplotlib
!pip install numpy
!pip install tqdm

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import warnings
warnings.filterwarnings('ignore')

## 2.2 Carga y Procesamiento de PDFs

Cargamos todos los archivos PDF del directorio especificado usando LangChain.

In [ ]:
# Configurar rutas
PDF_DIR = "./input/pdfs/"
OUTPUT_DIR = "./output/"

# Crear directorio de salida si no existe
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_pdfs_from_directory(pdf_dir):
    """
    Carga todos los PDFs desde un directorio usando PyPDFLoader
    """
    pdf_files = glob.glob(os.path.join(pdf_dir, "*.md"))  # Los PDFs están en formato .md
    
    if not pdf_files:
        print(f"⚠️  No se encontraron archivos .md en {pdf_dir}")
        return []
    
    all_documents = []
    
    print(f"\n{'='*60}")
    print(f"CARGANDO {len(pdf_files)} DOCUMENTOS PDF")
    print(f"{'='*60}\n")
    
    for pdf_path in tqdm(pdf_files, desc="Cargando PDFs"):
        try:
            # Leer el contenido del archivo .md
            with open(pdf_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            # Crear un documento con metadata
            doc = {
                'content': content,
                'metadata': {
                    'source': os.path.basename(pdf_path),
                    'path': pdf_path
                }
            }
            all_documents.append(doc)
            
        except Exception as e:
            print(f"Error cargando {pdf_path}: {e}")
    
    print(f"\n✓ {len(all_documents)} documentos cargados exitosamente")
    return all_documents

# Cargar documentos
documents = load_pdfs_from_directory(PDF_DIR)

if documents:
    print(f"\nEjemplo del primer documento:")
    print(f"  Fuente: {documents[0]['metadata']['source']}")
    print(f"  Longitud: {len(documents[0]['content'])} caracteres")
    print(f"  Primeros 200 caracteres: {documents[0]['content'][:200]}...")

## 2.3 Fragmentación del Texto (Chunking)

Dividimos los documentos en fragmentos (chunks) de tamaño razonable que preserven coherencia semántica.

In [ ]:
def chunk_documents(documents, chunk_size=500, chunk_overlap=50):
    """
    Fragmenta los documentos en chunks usando RecursiveCharacterTextSplitter
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    all_chunks = []
    
    print(f"\n{'='*60}")
    print(f"FRAGMENTANDO DOCUMENTOS")
    print(f"Tamaño de chunk: {chunk_size}, Overlap: {chunk_overlap}")
    print(f"{'='*60}\n")
    
    for doc in tqdm(documents, desc="Fragmentando"):
        chunks = text_splitter.split_text(doc['content'])
        
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                'text': chunk,
                'source': doc['metadata']['source'],
                'chunk_id': i
            })
    
    print(f"\n✓ {len(all_chunks)} fragmentos creados")
    return all_chunks

# Fragmentar documentos
chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=50)

if chunks:
    print(f"\nEjemplo del primer chunk:")
    print(f"  Fuente: {chunks[0]['source']}")
    print(f"  Chunk ID: {chunks[0]['chunk_id']}")
    print(f"  Longitud: {len(chunks[0]['text'])} caracteres")
    print(f"  Contenido: {chunks[0]['text'][:200]}...")

## 2.4 Configuración de Modelos de Sentence Transformers

Configuramos los 4 modelos especificados en el taller para comparar su desempeño.

In [ ]:
# Configuración de modelos según el taller
MODELS_CONFIG = {
    'all-mpnet-base-v2': {
        'name': 'all-mpnet-base-v2',
        'dimensions': 768,
        'description': 'Alta precisión (monolingüe)'
    },
    'all-MiniLM-L6-v2': {
        'name': 'all-MiniLM-L6-v2',
        'dimensions': 384,
        'description': 'Rápido y eficiente (monolingüe)'
    },
    'paraphrase-multilingual-MiniLM-L12-v2': {
        'name': 'paraphrase-multilingual-MiniLM-L12-v2',
        'dimensions': 384,
        'description': 'Búsqueda multilingüe'
    },
    'BAAI/bge-base-en': {
        'name': 'BAAI/bge-base-en',
        'dimensions': 768,
        'description': 'Recuperación y eficiencia'
    }
}

def load_models():
    """
    Carga todos los modelos configurados
    """
    models = {}
    
    print(f"\n{'='*60}")
    print("CARGANDO MODELOS DE SENTENCE TRANSFORMERS")
    print(f"{'='*60}\n")
    
    for model_key, config in MODELS_CONFIG.items():
        print(f"Cargando: {config['name']}")
        print(f"  Descripción: {config['description']}")
        print(f"  Dimensiones: {config['dimensions']}")
        
        try:
            model = SentenceTransformer(config['name'])
            models[model_key] = {
                'model': model,
                'config': config
            }
            print(f"  ✓ Cargado exitosamente\n")
        except Exception as e:
            print(f"  ✗ Error cargando modelo: {e}\n")
    
    return models

# Cargar modelos
models = load_models()
print(f"{'='*60}")
print(f"✓ {len(models)} modelos cargados correctamente")
print(f"{'='*60}")

## 2.5 Generación de Embeddings

Generamos embeddings para todos los fragmentos usando cada uno de los modelos.

In [ ]:
def generate_embeddings(chunks, models):
    """
    Genera embeddings para todos los chunks usando cada modelo
    """
    embeddings_by_model = {}
    chunk_texts = [chunk['text'] for chunk in chunks]
    
    print(f"\n{'='*60}")
    print("GENERANDO EMBEDDINGS")
    print(f"{'='*60}\n")
    
    for model_key, model_data in models.items():
        print(f"Generando embeddings con: {model_data['config']['name']}")
        
        model = model_data['model']
        embeddings = model.encode(chunk_texts, show_progress_bar=True, batch_size=32)
        
        embeddings_by_model[model_key] = {
            'embeddings': embeddings,
            'config': model_data['config']
        }
        
        print(f"  ✓ Shape: {embeddings.shape}")
        print(f"  ✓ {len(embeddings)} embeddings generados\n")
    
    return embeddings_by_model

# Generar embeddings
embeddings_by_model = generate_embeddings(chunks, models)

print(f"{'='*60}")
print(f"✓ Embeddings generados para todos los modelos")
print(f"{'='*60}")

## 2.6 Consulta de Similitud Semántica

Dada una oración de entrada, encontramos el fragmento más similar usando similitud coseno.

In [ ]:
def semantic_search(query, chunks, embeddings_by_model, models, top_k=1):
    """
    Busca los fragmentos más similares a una consulta usando similitud coseno
    """
    results = {}
    
    print(f"\n{'='*60}")
    print(f"BÚSQUEDA SEMÁNTICA")
    print(f"Query: '{query}'")
    print(f"{'='*60}\n")
    
    for model_key, emb_data in embeddings_by_model.items():
        model = models[model_key]['model']
        config = emb_data['config']
        
        # Generar embedding de la consulta
        query_embedding = model.encode([query])
        
        # Calcular similitud coseno
        similarities = cosine_similarity(query_embedding, emb_data['embeddings'])[0]
        
        # Obtener top-k fragmentos más similares
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results[model_key] = {
            'top_chunks': [(idx, similarities[idx], chunks[idx]) for idx in top_indices],
            'query_embedding': query_embedding,
            'config': config
        }
        
        print(f"Modelo: {config['name']}")
        print(f"  Dimensiones: {config['dimensions']}")
        
        for rank, (idx, sim, chunk) in enumerate(results[model_key]['top_chunks'], 1):
            print(f"\n  Rank {rank}:")
            print(f"    Similitud: {sim:.4f}")
            print(f"    Fuente: {chunk['source']}")
            print(f"    Fragmento: {chunk['text'][:150]}...")
        
        print()
    
    return results

# Definir una consulta de ejemplo
query_text = "¿Qué es la acreditación universitaria?"

# Realizar búsqueda semántica
search_results = semantic_search(
    query=query_text,
    chunks=chunks,
    embeddings_by_model=embeddings_by_model,
    models=models,
    top_k=3
)

## 2.7 Visualización 2D con PCA

Visualizamos la oración de entrada y el fragmento más similar en un plano 2D usando PCA para cada modelo.

In [ ]:
def visualize_pca(query, search_results, embeddings_by_model, model_key):
    """
    Visualiza con PCA la consulta y el fragmento más similar
    """
    # Obtener datos del modelo
    emb_data = embeddings_by_model[model_key]
    config = emb_data['config']
    all_embeddings = emb_data['embeddings']
    
    # Obtener el mejor resultado
    top_result = search_results[model_key]['top_chunks'][0]
    best_chunk_idx = top_result[0]
    similarity_score = top_result[1]
    
    # Preparar datos para PCA
    query_embedding = search_results[model_key]['query_embedding']
    best_chunk_embedding = all_embeddings[best_chunk_idx].reshape(1, -1)
    
    # Combinar embeddings para PCA
    combined_embeddings = np.vstack([
        query_embedding,
        best_chunk_embedding
    ])
    
    # Aplicar PCA
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(combined_embeddings)
    
    # Crear visualización
    plt.figure(figsize=(10, 8))
    
    # Plot query
    plt.scatter(embeddings_2d[0, 0], embeddings_2d[0, 1], 
               c='red', s=200, marker='*', label='Query', zorder=3)
    
    # Plot best chunk
    plt.scatter(embeddings_2d[1, 0], embeddings_2d[1, 1], 
               c='green', s=200, marker='o', label='Mejor Fragmento', zorder=3)
    
    # Anotaciones
    plt.annotate('Query', xy=(embeddings_2d[0, 0], embeddings_2d[0, 1]),
                xytext=(10, 10), textcoords='offset points', 
                fontsize=10, fontweight='bold')
    
    plt.annotate(f'Fragmento\n(sim: {similarity_score:.3f})', 
                xy=(embeddings_2d[1, 0], embeddings_2d[1, 1]),
                xytext=(10, -20), textcoords='offset points', 
                fontsize=10, fontweight='bold')
    
    # Línea conectando query y resultado
    plt.plot([embeddings_2d[0, 0], embeddings_2d[1, 0]], 
            [embeddings_2d[0, 1], embeddings_2d[1, 1]], 
            'b--', alpha=0.5, linewidth=1)
    
    plt.title(f'Visualización PCA - {config["name"]}\\n'\
             f'Query: "{query[:50]}..."', fontsize=12)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Guardar figura
    filename = f"{OUTPUT_DIR}{model_key.replace('/', '_')}_pca.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Gráfico guardado: {filename}")

# Visualizar para todos los modelos
print(f"\n{'='*60}")
print("VISUALIZACIONES PCA")
print(f"{'='*60}\n")

for model_key in models.keys():
    print(f"\nVisualizando: {embeddings_by_model[model_key]['config']['name']}")
    visualize_pca(query_text, search_results, embeddings_by_model, model_key)

## 2.8 Comparación Final de Modelos

Resumen y análisis comparativo de todos los modelos evaluados.

In [ ]:
print("\n" + "="*80)
print("RESUMEN DE COMPARACIÓN DE MODELOS")
print("="*80)

# Crear tabla comparativa
comparison_data = []

for model_key, results in search_results.items():
    config = results['config']
    best_sim = results['top_chunks'][0][1]
    
    comparison_data.append({
        'Modelo': config['name'],
        'Dimensiones': config['dimensions'],
        'Descripción': config['description'],
        'Similitud Máxima': f"{best_sim:.4f}"
    })

# Mostrar tabla
print("\n📊 TABLA COMPARATIVA:\n")
print(f"{'Modelo':<45} {'Dim':<6} {'Similitud':<10} {'Descripción'}")
print("-" * 100)

for data in comparison_data:
    print(f"{data['Modelo']:<45} {data['Dimensiones']:<6} {data['Similitud Máxima']:<10} {data['Descripción']}")

print("\n" + "="*80)
print("✅ CONCLUSIONES:")
print("="*80)
print("  1. Todos los modelos generaron embeddings exitosamente")
print("  2. La similitud coseno se calculó para cada modelo")
print("  3. Las visualizaciones PCA muestran la relación espacial entre query y resultados")
print("  4. Los modelos multilingües (paraphrase-multilingual) funcionan bien con español")
print("  5. Los modelos monolingües (all-mpnet, all-MiniLM) también capturan semántica")
print("  6. BAAI/bge-base-en es eficiente para recuperación de información")

print(f"\n💾 Visualizaciones guardadas en: {OUTPUT_DIR}")
print("="*80)